# AI Resume Optimizer

An intelligent resume optimization tool that tailors your resume for specific job descriptions using AI.

## How It Works
1. Run all cells in order
2. Mount Google Drive when prompted
3. Enter your Gemini API key
4. Upload or paste your resume
5. Paste a job description
6. Get an AI-optimized resume tailored to the job

## Features
- PDF resume upload or paste text
- AI-powered job description analysis
- Smart clarifying questions
- Resume optimization with keyword matching
- Save optimized resumes by category (ML Engineer, Data Scientist, etc.)
- Resume library stored on Google Drive

In [ ]:
# Install dependencies
!pip install -q gradio google-generativeai PyMuPDF

import gradio as gr
import google.generativeai as genai
import fitz  # PyMuPDF
import json
import os
import re
from datetime import datetime
from typing import Dict, List, Optional

print("All dependencies installed and imported successfully!")

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Set up data directory
DRIVE_PATH = "/content/drive/MyDrive/AI_Resume_Optimizer"
os.makedirs(f"{DRIVE_PATH}/resumes", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/history", exist_ok=True)

print(f"Data directory: {DRIVE_PATH}")
print("Directories created successfully!")

In [ ]:
# Configuration
GEMINI_MODEL = "gemini-1.5-flash"

CATEGORIES = [
    "ML Engineer",
    "Data Scientist",
    "AI Engineer",
    "Computer Vision",
    "Software Engineer",
    "Custom"
]

print(f"Model: {GEMINI_MODEL}")
print(f"Categories: {', '.join(CATEGORIES)}")
print("Configuration loaded!")

In [ ]:
# PDF Parsing and AI Engine

def parse_pdf(file_path: str) -> str:
    """Extract text from an uploaded PDF file."""
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text.strip()


class ResumeAI:
    """Gemini-powered resume optimization engine."""

    def __init__(self, api_key: str):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(GEMINI_MODEL)

    def analyze_job_description(self, jd_text: str) -> Dict:
        """Analyze a job description and extract structured requirements."""
        prompt = f"""Analyze this job description and extract structured information.
Return ONLY valid JSON with this exact structure:
{{
    "job_title": "the job title",
    "company": "company name if mentioned",
    "must_have_skills": ["skill1", "skill2"],
    "nice_to_have_skills": ["skill1", "skill2"],
    "key_responsibilities": ["resp1", "resp2"],
    "keywords": ["keyword1", "keyword2"],
    "experience_level": "entry/mid/senior",
    "summary": "2-3 sentence summary of what they're looking for"
}}

Job Description:
{jd_text}"""
        response = self.model.generate_content(prompt)
        text = response.text.strip()
        text = re.sub(r'^```json\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
        return json.loads(text)

    def generate_questions(self, resume_text: str, jd_analysis: Dict) -> List[Dict]:
        """Generate clarifying questions based on resume-JD gaps."""
        prompt = f"""You are a career coach. Compare this resume against job requirements and generate 2-3 targeted questions.

Resume:
{resume_text}

Job Requirements:
- Must have: {', '.join(jd_analysis.get('must_have_skills', []))}
- Nice to have: {', '.join(jd_analysis.get('nice_to_have_skills', []))}
- Key responsibilities: {', '.join(jd_analysis.get('key_responsibilities', []))}

Return ONLY valid JSON array:
[
    {{"id": 1, "question": "the question", "why": "brief reason this matters"}},
    {{"id": 2, "question": "the question", "why": "brief reason this matters"}}
]

Focus on skills gaps that could be addressed by rephrasing experience, which experiences to emphasize, and specific achievements to highlight."""
        response = self.model.generate_content(prompt)
        text = response.text.strip()
        text = re.sub(r'^```json\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
        return json.loads(text)

    def optimize_resume(self, resume_text: str, jd_analysis: Dict, answers: List[Dict]) -> str:
        """Generate an optimized resume tailored to the job description."""
        answers_text = "\n".join([f"Q: {a.get('question', '')} A: {a.get('answer', '')}" for a in answers])

        prompt = f"""You are an expert resume writer. Rewrite this resume to be optimized for the target job.

ORIGINAL RESUME:
{resume_text}

TARGET JOB:
Title: {jd_analysis.get('job_title', 'Unknown')}
Key Skills: {', '.join(jd_analysis.get('must_have_skills', []))}
Responsibilities: {', '.join(jd_analysis.get('key_responsibilities', []))}
Keywords: {', '.join(jd_analysis.get('keywords', []))}

ADDITIONAL CONTEXT FROM CANDIDATE:
{answers_text}

RULES:
1. Keep all factual information accurate - do NOT invent experience
2. Reword and reorganize to highlight relevant skills and experience
3. Naturally incorporate keywords from the job description
4. Tailor the professional summary to this specific role
5. Emphasize achievements that align with job requirements
6. Use strong action verbs and quantify achievements
7. Output the complete optimized resume as clean text"""

        response = self.model.generate_content(prompt)
        return response.text


# Global AI engine instance (initialized when API key is set)
ai_engine = None
print("PDF parser and AI engine defined!")

In [ ]:
# Resume Storage Manager

class ResumeStorage:
    """Manages saving and loading optimized resumes on Google Drive."""

    def __init__(self, base_path: str):
        self.base_path = base_path

    def save(self, category: str, optimized_text: str, job_title: str, company: str) -> str:
        """Save an optimized resume under a category folder."""
        cat_dir = os.path.join(self.base_path, "resumes", category.lower().replace(" ", "_"))
        os.makedirs(cat_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_title = job_title.replace(" ", "_")[:30]
        filename = f"{timestamp}_{safe_title}.txt"
        filepath = os.path.join(cat_dir, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(optimized_text)

        # Append to history log
        history_path = os.path.join(self.base_path, "history", "log.json")
        history = []
        if os.path.exists(history_path):
            with open(history_path, "r") as f:
                history = json.load(f)

        history.append({
            "timestamp": datetime.now().isoformat(),
            "category": category,
            "job_title": job_title,
            "company": company,
            "filename": filename
        })

        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        return filename

    def list_by_category(self) -> Dict:
        """List all saved resumes grouped by category."""
        resumes_dir = os.path.join(self.base_path, "resumes")
        categories = {}
        if not os.path.exists(resumes_dir):
            return categories

        for cat in sorted(os.listdir(resumes_dir)):
            cat_path = os.path.join(resumes_dir, cat)
            if os.path.isdir(cat_path):
                files = []
                for fname in sorted(os.listdir(cat_path), reverse=True):
                    fpath = os.path.join(cat_path, fname)
                    with open(fpath, "r", encoding="utf-8") as fh:
                        files.append({"name": fname, "content": fh.read()})
                categories[cat] = files

        return categories

    def get_history(self) -> List[Dict]:
        """Load optimization history."""
        history_path = os.path.join(self.base_path, "history", "log.json")
        if os.path.exists(history_path):
            with open(history_path, "r") as f:
                return json.load(f)
        return []


storage = ResumeStorage(DRIVE_PATH)
print(f"Storage manager ready. Saving to: {DRIVE_PATH}")

In [ ]:
# Application State

class AppState:
    """Manages current session state."""
    def __init__(self):
        self.resume_text = ""
        self.jd_analysis = None
        self.questions = []

state = AppState()
print("Application state initialized!")